In [ ]:
#HÜCRE 1
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:

#HÜCRE 5
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
print(os.environ.get('CUDA_LAUNCH_BLOCKING'))

1


In [ ]:
#HÜCRE 2
!pip install pandas==0.24.0 -q
!pip install bert_score evaluate tokenizers sacrebleu


  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'pandas' when getting requirements to build wheel


In [ ]:
#HÜCRE 3
import torch
!pip install zemberek-python
!pip install jpype1
!pip install torch-scatter -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install torch-geometric
print("\nKurulumlar tamamlandı. Zemberek'in Java'yı bulabilmesi için Runtime'ı yeniden başlatmanız gerekebilir.")

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu130.html
Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu130.html

Kurulumlar tamamlandı. Zemberek'in Java'yı bulabilmesi için Runtime'ı yeniden başlatmanız gerekebilir.


In [ ]:
#HÜCRE 4
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Cihaz: {device}")

Cihaz: cuda


In [ ]:
#HÜCRE 6

from google.colab import drive
import os

# 1. Drive'ı bağla (Zaten bağlıysa hata vermez, merak etme)
drive.mount('/content/drive')

# 2. Yolu tanımla (Görüntündeki bitişik MyDrive formatında)
CHECKPOINT_DIR = "/content/drive/MyDrive/Tez_Checkpoints"

# 3. KONTROL ET: Bu klasör gerçekten Drive'da var mı?
if os.path.exists(CHECKPOINT_DIR):
    print("BAĞLANTI TAMAM: Drive klasörüne erişildi.")
    files = os.listdir(CHECKPOINT_DIR)
    if files:
        print(f"Klasördeki mevcut dosyalar: {files}")
    else:
        print("Klasör şu an boş (İlk kayıt bekleniyor).")
else:
    print("HATA: Klasör bulunamadı! Lütfen akademik hesabınızın Drive'ında 'Tez_Checkpoints' isimli bir kısayol olduğundan emin olun.")
    # Kısayol yoksa yerel diskte yanlışlıkla oluşmasını engellemek için oluşturuyoruz
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print("ℹ️ Klasör yeni oluşturuldu. Lütfen Drive web sayfasından kontrol edin.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BAĞLANTI TAMAM: Drive klasörüne erişildi.
Klasördeki mevcut dosyalar: ['EN2TR_Standart_N6', 'SONUÇLAR', 'EN2TR_Standart_N3', 'WMT18_Standart_N3', 'WMT18_Standart_N6', 'EN2TR_Dual_N6', 'EN2TR_Dual_N6_withoutMD_and_FusionGate', 'EN2TR_CrossAttention', 'WMT18_CrossAttention_N6', 'EN2TR_FusionGate', 'EN2TR_GCN', 'EN2TR_Additive', 'EN2TR_ConcatCross', 'EN2TR_Standart_Seed42', 'EN2TR_CrossAttention_AltKümeler', 'EN2TR_Standart_AltKümeler', 'EN2TR_Standart_AltKümeler10']


In [ ]:
#HÜCRE 7

import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
#HÜCRE 8

import numpy as np
import copy
import time
from pathlib import Path

from tqdm import tqdm_notebook, tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import Dataset, DataLoader, Subset
from torch.utils.data.dataset import random_split

from transformer.model import Transformer, create_masks
#from transformer.batch import *

tqdm.pandas()

# Show better CUDA error messages
#!export CUDA_LAUNCH_BLOCKING=1

In [ ]:
#HÜCRE 9
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from nltk.tokenize import wordpunct_tokenize
from torch import optim
from torch.autograd import Variable
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader, Subset
from nltk.tokenize import word_tokenize
import re
import html
from tokenizers import Tokenizer, models, pre_tokenizers, trainers, decoders
from tokenizers.normalizers import Lowercase, NFD, StripAccents, Sequence

In [ ]:
#HÜCRE 10

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [ ]:
# YENİ KONFİGÜRASYON HÜCRESİ — Tüm eğitimleri buradan yönet

# === DEĞİŞTİRİLECEK İKİ SATIR ===
DATASET = "EN2TR"           # "EN2TR" veya "WMT18"
USE_DUAL_ENCODER = True    # False = Standart, True = Dual

# === Otomatik türetilen yollar (DOKUNMA) ===
import os

DATA_DIR       = f'/content/drive/MyDrive/Tez_Veri/{DATASET}'
MODE_SUFFIX    = "Dual" if USE_DUAL_ENCODER else "Standart"
CHECKPOINT_DIR = f'/content/drive/MyDrive/Tez_Checkpoints/EN2TR_CrossAttention/EN2TR_ParallelCross/{DATASET}_{MODE_SUFFIX}'

# Veri dosyaları
if DATASET == "EN2TR":
    TOKENIZER_FILE = f'{DATA_DIR}/bpe_tokenizer_en2tr.json'
    MORPH_VOCAB    = f'{DATA_DIR}/morph_vocab_en2tr.pkl'
else:  # WMT18
    TOKENIZER_FILE = f'{DATA_DIR}/bpe_tokenizer.json'
    MORPH_VOCAB    = f'{DATA_DIR}/morph_vocab.pkl'

TRAIN_PT = f'{DATA_DIR}/processed_train_graph_data.pt'
VALID_PT = f'{DATA_DIR}/processed_valid_graph_data.pt'
TEST_PT  = f'{DATA_DIR}/processed_test_graph_data.pt'

# Checkpoint klasörü hazırla
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"📂 Dataset       : {DATASET}")
print(f"🏗  Model Mode    : {MODE_SUFFIX}")
print(f"💾 Data Dir      : {DATA_DIR}")
print(f"📍 Checkpoint    : {CHECKPOINT_DIR}")
print(f"📄 Tokenizer     : {TOKENIZER_FILE}")
print(f"📄 Morph Vocab   : {MORPH_VOCAB}")

# Dosyaların var olduğunu doğrula
for f in [TOKENIZER_FILE, MORPH_VOCAB, TRAIN_PT, VALID_PT, TEST_PT]:
    status = "✅" if os.path.exists(f) else "❌ EKSİK"
    print(f"   {status}  {os.path.basename(f)}")

📂 Dataset       : EN2TR
🏗  Model Mode    : Dual
💾 Data Dir      : /content/drive/MyDrive/Tez_Veri/EN2TR
📍 Checkpoint    : /content/drive/MyDrive/Tez_Checkpoints/EN2TR_CrossAttention/EN2TR_ParallelCross/EN2TR_Dual
📄 Tokenizer     : /content/drive/MyDrive/Tez_Veri/EN2TR/bpe_tokenizer_en2tr.json
📄 Morph Vocab   : /content/drive/MyDrive/Tez_Veri/EN2TR/morph_vocab_en2tr.pkl
   ✅  bpe_tokenizer_en2tr.json
   ✅  morph_vocab_en2tr.pkl
   ✅  processed_train_graph_data.pt
   ✅  processed_valid_graph_data.pt
   ✅  processed_test_graph_data.pt


In [ ]:
# HÜCRE 13 (GÜNCELLENMİŞ)
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(TOKENIZER_FILE)
print(f"✅ Tokenizer yüklendi: {TOKENIZER_FILE}")
print(f"   Vocab size: {tokenizer.get_vocab_size()}")

✅ Tokenizer yüklendi: /content/drive/MyDrive/Tez_Veri/EN2TR/bpe_tokenizer_en2tr.json
   Vocab size: 40000


In [ ]:
# HÜCRE 14: VERİ YÜKLEME (NİHAİ, EN GÜVENİLİR VE <EOS> DÜZELTMELİ YÖNTEM)

# Gerekli importlar
from torch.utils.data import DataLoader, Dataset
from torch_geometric.data import Data, Batch
from torch.nn.utils.rnn import pad_sequence
import torch

# --- Konfigürasyon Değişkenleri ---
BATCH_SIZE = 32
MAX_LEN = 100
PAD_IDX = 0
SOS_IDX = 2
EOS_IDX = 3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Dataset Sınıfı ---
class FinalDataset(Dataset):
    def __init__(self, pt_file_path):
        print(f"Loading data from '{os.path.basename(pt_file_path)}'...")
        self.data_list = torch.load(pt_file_path, weights_only=False)
        self.data_list = [
            item for item in self.data_list
            if item['src_bpe'].numel() > 0 and item['trg_bpe'].numel() > 0
        ]
        print(f"   Loaded: {len(self.data_list)} examples")

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        return self.data_list[idx]


def final_collate_fn(batch_of_dicts, pad_idx):
    src_bpe_list = [item['src_bpe'] for item in batch_of_dicts]

    # SOS başa + EOS sona injection
    trg_bpe_list = [
        torch.cat([
            torch.tensor([SOS_IDX], dtype=torch.long),
            item['trg_bpe'].clone().detach(),
            torch.tensor([EOS_IDX], dtype=torch.long)
        ])
        for item in batch_of_dicts
    ]

    graph_data_list       = [item['graph_data']       for item in batch_of_dicts]
    src_bpe_word_ids_list = [item['src_bpe_word_ids'] for item in batch_of_dicts]

    padded_src_bpe          = pad_sequence(src_bpe_list,          batch_first=True, padding_value=pad_idx)
    padded_trg_bpe          = pad_sequence(trg_bpe_list,          batch_first=True, padding_value=pad_idx)
    padded_src_bpe_word_ids = pad_sequence(src_bpe_word_ids_list, batch_first=True, padding_value=0)
    batched_graph_data      = Batch.from_data_list(graph_data_list)

    return padded_src_bpe, padded_src_bpe_word_ids, batched_graph_data, padded_trg_bpe

# --- DataLoader'ları Oluştur (Drive'dan) ---
print("\nCreating DataLoaders from Drive...")
train_dataset = FinalDataset(TRAIN_PT)
valid_dataset = FinalDataset(VALID_PT)
test_dataset  = FinalDataset(TEST_PT)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=lambda b: final_collate_fn(b, PAD_IDX),
                          num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=lambda b: final_collate_fn(b, PAD_IDX),
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=lambda b: final_collate_fn(b, PAD_IDX),
                          num_workers=2, pin_memory=True)

print("✅ DataLoaders ready")


Creating DataLoaders from Drive...
Loading data from 'processed_train_graph_data.pt'...
   Loaded: 467034 examples
Loading data from 'processed_valid_graph_data.pt'...
   Loaded: 3000 examples
Loading data from 'processed_test_graph_data.pt'...
   Loaded: 3000 examples
✅ DataLoaders ready


In [ ]:

# HÜCRE 15: Quick check — bir validation batch'inden trg_bpe'yi incele
sample = next(iter(valid_loader))
v_src_bpe, v_src_bpe_word_ids, v_src_graph, v_trg_bpe = sample

# trg_bpe collate'ten EOS'lu çıkıyor, ama SOS başta mı?
print(f"trg_bpe[0] ilk 10 token : {v_trg_bpe[0][:10].tolist()}")
print(f"İlk token              : {v_trg_bpe[0][0].item()}  (2 olmalı = SOS)")
print(f"Bilinen ID'ler         : PAD=0, UNK=1, SOS=2, EOS=3")

trg_bpe[0] ilk 10 token : [2, 41, 268, 144, 2980, 164, 7933, 28, 13348, 16]
İlk token              : 2  (2 olmalı = SOS)
Bilinen ID'ler         : PAD=0, UNK=1, SOS=2, EOS=3


In [ ]:
# HÜCRE 15 (GÜNCELLENMİŞ): MODEL OLUŞTURMA
import pickle

vocab_size = tokenizer.get_vocab_size()
print(f"BPE Vocabulary Size: {vocab_size}")

with open(MORPH_VOCAB, 'rb') as f:
    morph_vocab = pickle.load(f)
morph_vocab_size = len(morph_vocab)
print(f"Morpheme Vocabulary Size: {morph_vocab_size}")

# Hiperparametreler
HEADS   = 8
N       = 6
D_MODEL = 512
DROPOUT = 0.25
MAX_LEN = 5000

import torch
def _max_morph_id(ds):
    base = ds.dataset if hasattr(ds, 'dataset') else ds   # Subset (düşük-kaynak) için
    return max(int(it['graph_data'].x.max()) for it in base.data_list)

mx = max(_max_morph_id(train_dataset),
         _max_morph_id(valid_dataset),
         _max_morph_id(test_dataset))
morph_vocab_size = max(len(morph_vocab), mx + 1)
print("veri max morph id:", mx, "| morph_vocab_size ->", morph_vocab_size)

model = Transformer(
    vocab_size       = vocab_size,
    d_model          = D_MODEL,
    max_len          = MAX_LEN,
    heads            = HEADS,
    dropout          = DROPOUT,
    N                = N,
    morph_vocab_size = morph_vocab_size,
    use_dual_encoder = USE_DUAL_ENCODER,
    fusion           = "cross_attn",

).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n✅ Model oluşturuldu")
print(f"   Mod: {'Dual Encoder' if USE_DUAL_ENCODER else 'Standart Transformer (Baseline)'}")
print(f"   Toplam parametre: {total_params:,}")

BPE Vocabulary Size: 40000
Morpheme Vocabulary Size: 14421
veri max morph id: 14427 | morph_vocab_size -> 14428

✅ Model oluşturuldu
   Mod: Dual Encoder
   Toplam parametre: 107,239,488


In [ ]:
# HÜCRE 17 (GÜNCELLENMİŞ: PARAMETRE GRUPLAMA)
def count_parameters_detailed(model):
    embed_params = sum(p.numel() for p in model.bpe_embed.parameters() if p.requires_grad)

    # 1. Kodlayıcı Parametreleri
    enc_bpe_params = sum(p.numel() for p in model.encoder_bpe.parameters() if p.requires_grad)

    # 2. Kodlayıcı Parametreleri
    enc_graph_params = sum(p.numel() for p in model.graph_encoder.parameters() if p.requires_grad)

    dec_layers_params = sum(p.numel() for p in model.decoder_layers.parameters() if p.requires_grad)
    out_params = sum(p.numel() for p in model.out.parameters() if p.requires_grad)
    norm_params = sum(p.numel() for p in model.norm.parameters() if p.requires_grad)

    dec_total_params = dec_layers_params + out_params + norm_params
    total_params = embed_params + enc_bpe_params + enc_graph_params + dec_total_params

    print("="*30)
    print("MODEL COMPLEXITY ANALYSIS (DUAL ENCODER)")
    print("="*30)
    print(f"BPE Embeddings Params: {embed_params:,}")
    print(f"BPE Encoder Params   : {enc_bpe_params:,}")
    print(f"Graph Encoder Params : {enc_graph_params:,}")
    print(f"Decoder Params       : {dec_total_params:,}")
    print("-" * 30)
    print(f"TOTAL Trainable      : {total_params:,}")
    print("="*30)

count_parameters_detailed(model)

MODEL COMPLEXITY ANALYSIS (DUAL ENCODER)
BPE Embeddings Params: 20,480,000
BPE Encoder Params   : 18,914,304
Graph Encoder Params : 15,791,616
Decoder Params       : 52,048,960
------------------------------
TOTAL Trainable      : 107,234,880


In [ ]:
model.train()
sample = next(iter(train_loader))
src_bpe, src_bpe_word_ids, src_graph, trg_bpe = sample
src_bpe = src_bpe.to(device); src_graph = src_graph.to(device)
src_bpe_word_ids = src_bpe_word_ids.to(device); trg_bpe = trg_bpe.to(device)

out = model(src_graph, trg_bpe[:, :-1], src_bpe, src_bpe_word_ids, 0)
print(f"✅ Forward çalıştı. Çıktı boyutu: {out.shape}")
# Beklenen: [batch, trg_len-1, vocab_size]

✅ Forward çalıştı. Çıktı boyutu: torch.Size([32, 16, 40000])


In [ ]:
# Dual modeli yükle, gate ortalamasını ölç
model.eval()
with torch.no_grad():
    sample = next(iter(valid_loader))
    src_bpe, src_bpe_word_ids, src_graph, trg_bpe = sample
    src_bpe = src_bpe.to(device); src_graph = src_graph.to(device)
    src_bpe_word_ids = src_bpe_word_ids.to(device)
    B = src_bpe.size(0)
    src_mask, _ = create_masks(src_bpe, None, 0, device)

    bpe_x = model.pe(model.bpe_embed(src_bpe))
    for layer in model.encoder_bpe:
        bpe_x = layer(bpe_x, src_mask)
    bpe_x = model.encoder_norm(bpe_x)

    graph_out, _ = model.graph_encoder(src_graph, expected_batch_size=B)
    max_w = graph_out.size(1)
    safe = torch.clamp(src_bpe_word_ids, 0, max(0, max_w-1))
    gm = torch.gather(graph_out, 1, safe.unsqueeze(-1).expand(B, src_bpe.size(1), model.d_model))

    concat = torch.cat([bpe_x, gm], dim=-1)
    gate = torch.sigmoid(model.fusion_gate(concat))
    print(f"Gate ort: {gate.mean().item():.3f} (1.0=sadece BPE, 0.0=sadece graph)")
    print(f"BPE norm: {bpe_x.norm(dim=-1).mean().item():.2f}")
    print(f"Graph norm: {gm.norm(dim=-1).mean().item():.2f}")

AttributeError: 'Transformer' object has no attribute 'fusion_gate'

In [ ]:
# HÜCRE 18: Optimizasyon ve Öğrenme Oranı Zamanlayıcısı

class WarmupInverseSquareRootScheduler(torch.optim.lr_scheduler._LRScheduler):
    """
    "Attention is All You Need" makalesinde açıklanan öğrenme oranı zamanlayıcısı.

    Eğitimin başlarında öğrenme oranını 'warmup_steps' boyunca lineer olarak artırır,
    ardından model boyutuna ve adım sayısının ters kareköküne göre azaltır.
    """
    def __init__(self, optimizer, d_model, warmup_steps, last_epoch=-1):
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        # self.last_epoch, PyTorch scheduler'ının içindeki adım sayacıdır. 0'dan başlar.
        # Adım sayısını 1'den başlatmak için +1 ekliyoruz.
        step_num = self.last_epoch + 1

        # Formül: lr = d_model^(-0.5) * min(step_num^(-0.5), step_num * warmup_steps^(-1.5))
        arg1 = step_num ** -0.5
        arg2 = step_num * (self.warmup_steps ** -1.5)

        # d_model**-0.5 sabitini her bir başlangıç LR ile çarp
        return [
            base_lr * (self.d_model ** -0.5) * min(arg1, arg2)
            for base_lr in self.base_lrs
        ]

In [ ]:
# HÜCRE 14: Optimizasyon ve Öğrenme Oranı Zamanlayıcısı


criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)

optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)

WARMUP_STEPS = 8000
scheduler = WarmupInverseSquareRootScheduler(optimizer, d_model=D_MODEL, warmup_steps=WARMUP_STEPS)

print("Criterion, Optimizer ve Scheduler başarıyla oluşturuldu.")
print(f"Model boyutu (d_model): {D_MODEL}, Isınma adımları (warmup_steps): {WARMUP_STEPS}")

Criterion, Optimizer ve Scheduler başarıyla oluşturuldu.
Model boyutu (d_model): 512, Isınma adımları (warmup_steps): 8000


In [ ]:
# HÜCRE 19 (GÜNCELLENMİŞ: Ceza Kaldırıldı ve Görsel Hata Ayıklama Eklendi)
import sacrebleu
import torch
from torch_geometric.data import Batch

def calculate_epoch_bleu(loader, model, tokenizer, device):
    model.eval()
    preds, refs = [], []

    max_samples = 1500
    current_samples = 0
    printed_example = False # Ekrana 1 örnek yazdırmak için bayrak

    with torch.no_grad():
        for data in loader:
            if current_samples >= max_samples: break
            src_bpe, src_bpe_word_ids, src_graph, trg_bpe = data

            graph_list = src_graph.to_data_list() if hasattr(src_graph, 'to_data_list') else src_graph

            for i in range(src_bpe.size(0)):
                if current_samples >= max_samples: break

                single_graph = Batch.from_data_list([graph_list[i]]).to(device)
                single_src_bpe = src_bpe[i:i+1].to(device)
                single_word_ids = src_bpe_word_ids[i:i+1].to(device)

                pred_ids = model.generate(
                    src_graph=single_graph,
                    src_bpe=single_src_bpe,
                    src_bpe_word_ids=single_word_ids,
                    sos_idx=2,
                    eos_idx=3,
                    pad_idx=0,
                    max_len=100,
                    beam_size=3,
                    repetition_penalty=1.0   # ÇOK ÖNEMLİ: Çeviri için ceza 1.0 (etkisiz) olmalı!
                )

                # Üretilen ID'leri metne çevir ve temel noktalama boşluklarını temizle
                pred_text = tokenizer.decode(pred_ids, skip_special_tokens=True)
                pred_text = pred_text.replace(" .", ".").replace(" ,", ",").replace(" ?", "?").replace(" !", "!")
                preds.append(pred_text)

                # Hedef (Referans) metni hazırla ve temel noktalama boşluklarını temizle
                ref_text = tokenizer.decode(trg_bpe[i].cpu().tolist(), skip_special_tokens=True)
                ref_text = ref_text.replace(" .", ".").replace(" ,", ",").replace(" ?", "?").replace(" !", "!")
                refs.append(ref_text)

                # SADECE 1 KERE ÇALIŞACAK KISIM: Modelin ne yaptığını görelim!
                if not printed_example:
                    print(f"\n--- ÖRNEK ÇEVİRİ ---")
                    print(f"Hedef (Olması Gereken): {ref_text}")
                    print(f"Modelin Çıktısı      : {pred_text}")
                    print(f"--------------------\n")
                    printed_example = True

                current_samples += 1

    bleu = sacrebleu.corpus_bleu(preds, [refs])
    return bleu.score

In [ ]:
#HÜCRE 20

import time
import torch

def calculate_epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

In [ ]:
# HÜCRE 21: ÇİFT KODLAYICILI MODEL EĞİTİM FONKSİYONU
import os
import time
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

def train_dual_encoder(train_loader, valid_loader, model, epochs, optimizer, criterion, scheduler, device, pad_idx, start_epoch=1, eval_every=1):
    model.to(device)
    checkpoint_dir = CHECKPOINT_DIR #WMT_Standart, EN2TR_Dual, WMT_Dual

    # Klasör yoksa oluştur
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)

    metrics_csv = os.path.join(checkpoint_dir, "metrics_history.csv")

    # Geçmiş metrikleri yükle veya sıfırdan başlat
    if os.path.exists(metrics_csv) and start_epoch > 1:
        history = pd.read_csv(metrics_csv).to_dict(orient='list')
    else:
        history = {'epoch': [], 'train_loss': [], 'val_loss': [], 'bleu': [], 'duration': []}

    #best_bleu = 0.0 if not history['bleu'] else max([x for x in history['bleu'] if not np.isnan(x)])
    _vl = [x for x in history['val_loss'] if not np.isnan(x)]
    best_val_loss = float('inf') if not _vl else min(_vl)

    for epoch in range(start_epoch, epochs + 1):
        # 1. VRAM Sayacını Sıfırla (Sadece bu epoch'un yükünü ölçmek için)
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        epoch_start_time = time.time()

        # ==========================================
        # 1. EĞİTİM (TRAIN) AŞAMASI
        # ==========================================
        model.train()
        train_losses = []
        pbar = tqdm(train_loader, desc=f'Epoch {epoch:02d} [Train]', leave=False)

        for data in pbar:
            src_bpe, src_bpe_word_ids, src_graph, trg_bpe = data

            src_bpe = src_bpe.to(device)
            src_graph = src_graph.to(device)
            src_bpe_word_ids = src_bpe_word_ids.to(device)
            trg_bpe = trg_bpe.to(device)

            # Decoder'a gidecek input'u (son token olan EOS hariç) oluşturuyoruz
            trg_bpe_input = trg_bpe[:, :-1]

            optimizer.zero_grad()

            # forward fonksiyonu signature'ına tam uyumlu sıralama
            output = model(src_graph, trg_bpe_input, src_bpe, src_bpe_word_ids, pad_idx)

            loss = criterion(output.contiguous().view(-1, output.shape[-1]), trg_bpe[:, 1:].contiguous().view(-1))

            if torch.isnan(loss):
                print(f"❌ KRİTİK: Epoch {epoch} NaN! Bu batch atlanıyor...")
                continue # Durmak yerine atla, ağırlıkları bozma

            loss.backward()

            # --- SERT GRADYAN KIRPMA ---
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            scheduler.step()

            train_losses.append(loss.item())
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        # ==========================================
        # 2. DOĞRULAMA (VALIDATION) AŞAMASI
        # ==========================================
        model.eval()
        val_losses = []
        with torch.no_grad():
            for b in valid_loader:
                if b is None: continue

                v_src_bpe, v_src_bpe_word_ids, v_src_graph, v_trg_bpe = b

                v_src_bpe = v_src_bpe.to(device)
                v_src_graph = v_src_graph.to(device)
                v_src_bpe_word_ids = v_src_bpe_word_ids.to(device)
                v_trg_bpe = v_trg_bpe.to(device)

                v_trg_bpe_input = v_trg_bpe[:, :-1]

                v_out = model(v_src_graph, v_trg_bpe_input, v_src_bpe, v_src_bpe_word_ids, pad_idx)
                v_loss = criterion(v_out.contiguous().view(-1, v_out.shape[-1]), v_trg_bpe[:, 1:].contiguous().view(-1))
                val_losses.append(v_loss.item())

        # ==========================================
        # 3. EPOCH SONU HESAPLAMALARI VE METRİKLER
        # ==========================================
        avg_train_loss = np.mean(train_losses)
        avg_val_loss = np.mean(val_losses)

        if np.isnan(avg_val_loss): continue

        # Cosine Annealing için scheduler güncelleniyor (Artık val_loss almaz, epoch bazlı çalışır)


        # BLEU hesaplama (tokenizer değişkeninin globalde tanımlı olduğu varsayılıyor)
        #epoch_bleu = calculate_epoch_bleu(valid_loader, model, tokenizer, device)
        epoch_bleu = 0.0   # eğitim sırasında BLEU hesaplanmıyor (hız + generate'i atla)

        # Zaman Hesaplaması
        epoch_duration = time.time() - epoch_start_time
        epoch_mins = int(epoch_duration / 60)
        epoch_secs = int(epoch_duration - (epoch_mins * 60))

        # VRAM Hesaplaması
        if torch.cuda.is_available():
            max_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3) # Byte'ı GB'a çevir
        else:
            max_vram_gb = 0.0

        # ==========================================
        # 4. KAYIT (SAVING) İŞLEMLERİ
        # ==========================================
        history['epoch'].append(epoch)
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['bleu'].append(epoch_bleu)
        history['duration'].append(epoch_duration)
        pd.DataFrame(history).to_csv(metrics_csv, index=False)

        # En iyi modeli kaydet
        #if epoch_bleu > best_bleu:
        #  best_bleu = epoch_bleu
         # torch.save({
          #    'epoch': epoch,
           #   'model_state_dict': model.state_dict(),
#              'optimizer_state_dict': optimizer.state_dict(),
 #             'scheduler_state_dict': scheduler.state_dict(),
 #             'best_bleu': best_bleu,
  #        }, os.path.join(checkpoint_dir, "model_best.pth"))
   #       print(f"  ✅ Yeni en iyi model kaydedildi — BLEU: {best_bleu:.2f}")


        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_val_loss': best_val_loss,
            }, os.path.join(checkpoint_dir, "model_best.pth"))
            print(f"  ✅ Yeni en iyi model (val_loss) kaydedildi — val_loss: {best_val_loss:.4f}")

        # Son modeli her halükarda kaydet (Bağlantı kopmasına karşı tam sigorta)
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(), # Scheduler hafızası güvende
            'val_loss': avg_val_loss
        }, os.path.join(checkpoint_dir, "model_latest.pth"))

        tqdm.write(f"\nEpoch: {epoch:02d} | Süre: {epoch_mins}dk {epoch_secs}sn | Max VRAM: {max_vram_gb:.2f} GB")
        tqdm.write(f"    Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | BLEU: {epoch_bleu:.2f}\n")

    print("--- ✅ Eğitim Bitti ---")

In [ ]:
import torch
from torch_geometric.data import Batch
dev='cpu'

m = Transformer(vocab_size=vocab_size, d_model=D_MODEL, max_len=MAX_LEN, heads=HEADS,
                dropout=DROPOUT, N=N, morph_vocab_size=morph_vocab_size,
                use_dual_encoder=True, fusion="parallel_cross").to(dev)
crit = torch.nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)

batch = next(iter(train_loader))
src_bpe, src_bpe_word_ids, src_graph, trg_bpe = batch

# 1) eğitim adımı: forward + loss + backward
out = m(src_graph.to(dev), trg_bpe[:, :-1].to(dev),
        src_bpe.to(dev), src_bpe_word_ids.to(dev), 0)
loss = crit(out.reshape(-1, out.size(-1)), trg_bpe[:, 1:].reshape(-1).to(dev))
loss.backward()
print("TRAIN STEP OK | loss:", round(loss.item(), 4))

# 2) generate (epoch sonu değerlendirmesi — asıl şüpheli)
g0 = Batch.from_data_list([src_graph.to_data_list()[0]]).to(dev)
ids = m.generate(src_graph=g0, src_bpe=src_bpe[0:1].to(dev),
                 src_bpe_word_ids=src_bpe_word_ids[0:1].to(dev),
                 sos_idx=2, eos_idx=3, pad_idx=0,
                 max_len=50, beam_size=3, repetition_penalty=1.0)
print("GENERATE OK | len:", len(ids))

KeyboardInterrupt: 

In [ ]:
from torch.utils.data import DataLoader

# shuffle KAPALI tarama loader'ı (batch index sabit olsun)
scan_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=lambda b: final_collate_fn(b, PAD_IDX))

bozuk = 0
for bi, batch in enumerate(scan_loader):
    src_bpe, src_bpe_word_ids, src_graph, trg_bpe = batch
    B = src_bpe.size(0)
    G = int(src_graph.batch.max()) + 1
    checks = []
    if G != B:
        checks.append(f"G({G})!=B({B})")
    if int(src_graph.x.max()) >= morph_vocab_size:
        checks.append(f"morph_id {int(src_graph.x.max())} >= {morph_vocab_size}")
    if int(src_graph.x.min()) < 0:
        checks.append(f"morph_id negatif {int(src_graph.x.min())}")
    if int(src_bpe.max()) >= vocab_size:
        checks.append(f"src_id {int(src_bpe.max())} >= {vocab_size}")
    if int(trg_bpe.max()) >= vocab_size:
        checks.append(f"trg_id {int(trg_bpe.max())} >= {vocab_size}")
    if src_graph.x.size(0) != src_graph.word_batch.size(0):
        checks.append(f"node({src_graph.x.size(0)}) != word_batch({src_graph.word_batch.size(0)})")
    if int(src_graph.word_batch.max()) < 1:
        checks.append(f"word_batch.max={int(src_graph.word_batch.max())} (çok küçük)")
    if checks:
        print(f"⚠️ batch {bi}: " + " | ".join(checks))
        bozuk += 1
        if bozuk >= 5:
            print("... (ilk 5 anomali gösterildi)")
            break

print(f"Tarama bitti. Anomali sayısı: {bozuk}")

⚠️ batch 63: morph_id 14422 >= 14421
⚠️ batch 318: morph_id 14422 >= 14421
⚠️ batch 997: morph_id 14422 >= 14421
⚠️ batch 1238: morph_id 14422 >= 14421
⚠️ batch 1340: morph_id 14422 >= 14421
... (ilk 5 anomali gösterildi)
Tarama bitti. Anomali sayısı: 5


In [ ]:
# HÜCRE 21 (GÜNCELLENMİŞ)
import os
import torch

LATEST_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "model_latest.pth")
TOTAL_EPOCHS = 80

start_epoch = 1

if os.path.exists(LATEST_CHECKPOINT_PATH):
    print(f"🔄 Checkpoint yükleniyor: {LATEST_CHECKPOINT_PATH}")
    checkpoint = torch.load(LATEST_CHECKPOINT_PATH, map_location=device, weights_only=False)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    if 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        print("✅ Scheduler hafızası yüklendi.")

    start_epoch = checkpoint['epoch'] + 1
    print(f"✅ Eğitime {start_epoch}. epoch'tan devam.")
    print(f"📉 Mevcut LR: {optimizer.param_groups[0]['lr']:.6e}")
else:
    print(f"🆕 Checkpoint yok ({CHECKPOINT_DIR})")
    print("   Sıfırdan eğitime başlanıyor.")

train_dual_encoder(
    train_loader=train_loader,
    valid_loader=valid_loader,
    model=model,
    epochs=TOTAL_EPOCHS,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    device=device,
    pad_idx=PAD_IDX,
    start_epoch=start_epoch,
    eval_every=1
)

🆕 Checkpoint yok (/content/drive/MyDrive/Tez_Checkpoints/EN2TR_CrossAttention/EN2TR_ParallelCross/EN2TR_Dual)
   Sıfırdan eğitime başlanıyor.


  ✅ Yeni en iyi model (val_loss) kaydedildi — val_loss: 3.0731

Epoch: 01 | Süre: 54dk 42sn | Max VRAM: 3.95 GB
    Train Loss: 4.0217 | Val Loss: 3.0731 | BLEU: 0.00



  ✅ Yeni en iyi model (val_loss) kaydedildi — val_loss: 2.7637

Epoch: 02 | Süre: 54dk 20sn | Max VRAM: 3.96 GB
    Train Loss: 3.0245 | Val Loss: 2.7637 | BLEU: 0.00



  ✅ Yeni en iyi model (val_loss) kaydedildi — val_loss: 2.6090

Epoch: 03 | Süre: 54dk 5sn | Max VRAM: 4.00 GB
    Train Loss: 2.8079 | Val Loss: 2.6090 | BLEU: 0.00



  ✅ Yeni en iyi model (val_loss) kaydedildi — val_loss: 2.5228

Epoch: 04 | Süre: 52dk 13sn | Max VRAM: 4.01 GB
    Train Loss: 2.6796 | Val Loss: 2.5228 | BLEU: 0.00



  ✅ Yeni en iyi model (val_loss) kaydedildi — val_loss: 2.4704

Epoch: 05 | Süre: 52dk 25sn | Max VRAM: 3.99 GB
    Train Loss: 2.5944 | Val Loss: 2.4704 | BLEU: 0.00



  ✅ Yeni en iyi model (val_loss) kaydedildi — val_loss: 2.4232

Epoch: 06 | Süre: 54dk 23sn | Max VRAM: 3.95 GB
    Train Loss: 2.5322 | Val Loss: 2.4232 | BLEU: 0.00



  ✅ Yeni en iyi model (val_loss) kaydedildi — val_loss: 2.3784

Epoch: 07 | Süre: 54dk 25sn | Max VRAM: 3.95 GB
    Train Loss: 2.4808 | Val Loss: 2.3784 | BLEU: 0.00



  ✅ Yeni en iyi model (val_loss) kaydedildi — val_loss: 2.3544

Epoch: 08 | Süre: 54dk 29sn | Max VRAM: 3.98 GB
    Train Loss: 2.4380 | Val Loss: 2.3544 | BLEU: 0.00



Epoch 09 [Train]:  38%|███▊      | 5569/14595 [20:57<34:26,  4.37it/s, loss=2.5659]

In [ ]:
#HÜCRE 23
model.eval()
with torch.no_grad():
    sample = next(iter(valid_loader))
    src_bpe, src_bpe_word_ids, src_graph, trg_bpe = sample
    src_bpe          = src_bpe.to(device)
    src_graph        = src_graph.to(device)
    src_bpe_word_ids = src_bpe_word_ids.to(device)

    B = src_bpe.size(0)
    from transformer.model import create_masks
    src_mask, _ = create_masks(src_bpe, None, 0, device)

    bpe_x = model.pe(model.bpe_embed(src_bpe))
    for layer in model.encoder_bpe:
        bpe_x = layer(bpe_x, src_mask)

    graph_out_dense, _ = model.graph_encoder(src_graph, expected_batch_size=B)
    max_words = graph_out_dense.size(1)
    safe_ids  = torch.clamp(src_bpe_word_ids, 0, max(0, max_words - 1))
    safe_ids_exp = safe_ids.unsqueeze(-1).expand(B, src_bpe.size(1), model.d_model)
    graph_memory = torch.gather(graph_out_dense, 1, safe_ids_exp)

    concat = torch.cat([bpe_x, graph_memory], dim=-1)
    gate   = torch.sigmoid(model.fusion_gate(concat))

    print(f"Gate ort  : {gate.mean().item():.3f}  (1.0 = sadece BPE, 0.0 = sadece graph)")
    print(f"Gate min  : {gate.min().item():.3f}")
    print(f"Gate max  : {gate.max().item():.3f}")
    print(f"Gate std  : {gate.std().item():.3f}")
    print(f"BPE norm  : {bpe_x.norm(dim=-1).mean().item():.3f}")
    print(f"Graph norm: {graph_memory.norm(dim=-1).mean().item():.3f}")

Gate ort  : 0.562  (1.0 = sadece BPE, 0.0 = sadece graph)
Gate min  : 0.000
Gate max  : 1.000
Gate std  : 0.491
BPE norm  : 949.981
Graph norm: 13.416


In [ ]:
#HÜCRE 25
import matplotlib.pyplot as plt
import pandas as pd

# CSV'yi oku
df = pd.read_csv(f"{CHECKPOINT_DIR}/metrics_history.csv")

plt.figure(figsize=(14, 5))

# 1. Graf: Loss Değerleri
plt.subplot(1, 2, 1)
plt.plot(df['epoch'], df['train_loss'], label='Train Loss', color='black', linewidth=1.5, marker='o', markersize=2)
plt.plot(df['epoch'], df['val_loss'], label='Val Loss', color='blue', linewidth=1.5, marker='x', markersize=2)
plt.title('Training and Validation Loss', fontsize=12)
plt.xlabel('Epoch', fontsize=10)
plt.ylabel('Loss', fontsize=10)
plt.legend()
plt.grid(True, alpha=0.3)

# 2. Graf: BLEU Skoru
plt.subplot(1, 2, 2)
plt.plot(df['epoch'], df['bleu'], label='BLEU Score', color='red', linewidth=2, marker='^', markersize=3)
plt.title('Evaluation BLEU Score', fontsize=12)
plt.xlabel('Epoch', fontsize=10)
plt.ylabel('BLEU', fontsize=10)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{CHECKPOINT_DIR}/training_results_figure.png", dpi=300)
plt.show()


In [ ]:
#HÜCRE 26
def plot_bleu_by_length_ieee(results_df):
    plt.figure(figsize=(9, 6))

    # Renk paleti: IEEE Access genellikle sade ve kontrast renkler sever
    colors = ['#1f77b4', '#3498db', '#5dade2', '#85c1e9', '#aed6f1']

    bars = plt.bar(results_df['Range'], results_df['BLEU'],
                   color='#2c3e50', edgecolor='black', alpha=0.8, width=0.6)

    # Sütunların üzerine tam sayı değerlerini yazalım (Hakemlerin çok sevdiği bir detay)
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 0.5, yval,
                 ha='center', va='bottom', fontsize=11, fontweight='bold')

    plt.title('Impact of Sentence Length on Translation Quality', fontsize=14, fontweight='bold')
    plt.xlabel('Source Sentence Length (Number of Words)', fontsize=12)
    plt.ylabel('BLEU Score', fontsize=12)
    plt.ylim(0, max(results_df['BLEU']) + 5) # Üstte pay bırak
    plt.grid(axis='y', linestyle='--', alpha=0.6)

    # Makale için kaydet (Yüksek DPI)
    plt.tight_layout()
    plt.savefig("bleu_vs_length_ieee.png", dpi=300)
    plt.show()

In [ ]:
# HÜCRE 25 (GÜNCELLENMİŞ)
best_model_path = f"{CHECKPOINT_DIR}/model_best_weights.pth"

if os.path.exists(best_model_path):
    print("✨ En iyi model ağırlıkları yükleniyor...")
    best_weights = torch.load(best_model_path, map_location=device, weights_only=True)
    model.load_state_dict(best_weights)
    print("✅ Yüklendi.")
else:
    print("⚠️ En iyi model yok, mevcut ağırlıklarla devam ediliyor.")

In [ ]:
# HÜCRE 28: Çeviri Örneklerini Gösterme (DÜZELTİLMİŞ NİHAİ HAL)
import torch

# Eğer MAX_LEN globalde tanımlı değilse burada manuel bir sınır koyalım (örn: 100)
MAX_LEN = 100

def translate_and_show(loader, model, tokenizer, device, num_samples=5):
    """
    Çift kodlayıcılı model için son ve temiz çeviri gösterme fonksiyonu.
    """
    model.eval()
    samples_shown = 0

    print("\n--- Çeviri Örnekleri Üretiliyor ---")
    with torch.no_grad():
        for data in loader:
            if data is None:
                continue

            src_bpe, src_bpe_word_ids, src_graph, trg_bpe = data

            for i in range(src_bpe.shape[0]):
                if samples_shown >= num_samples:
                    break

                # Batch'ten i'inci örneği ayıkla ve cihaza gönder
                src_bpe_sentence = src_bpe[i:i+1].to(device)
                src_bpe_word_ids_sentence = src_bpe_word_ids[i:i+1].to(device)
                src_graph_sentence = src_graph[i].to(device)

                # ==========================================
                # DÜZELTME YAPILAN KISIM (Değişken isimleri eşleşti)
                # ==========================================
                pred_ids = model.generate(
                    src_graph=src_graph_sentence,               # <-- DÜZELTİLDİ
                    src_bpe=src_bpe_sentence,                   # <-- DÜZELTİLDİ
                    src_bpe_word_ids=src_bpe_word_ids_sentence, # <-- DÜZELTİLDİ
                    sos_idx=2,
                    eos_idx=3,
                    pad_idx=0,
                    max_len=100,
                    beam_size=3,
                    repetition_penalty=1.0
                )

                # ID dizilerini metne çevir
                input_text = tokenizer.decode(src_bpe[i].cpu().tolist(), skip_special_tokens=True)
                predicted_text = tokenizer.decode(pred_ids, skip_special_tokens=True) # <-- DÜZELTİLDİ
                target_text = tokenizer.decode(trg_bpe[i].cpu().tolist(), skip_special_tokens=True)

                print(f"🟦 KAYNAK (Input):  {input_text}")
                print(f"🟩 TAHMİN (Output): {predicted_text}")
                print(f"🟥 HEDEF (Target):  {target_text}")
                print("-" * 80)

                samples_shown += 1

            if samples_shown >= num_samples:
                break

# --- Fonksiyonu Çağırma ---
translate_and_show(
    loader=test_loader, # Veya valid_loader
    model=model,
    tokenizer=tokenizer,
    device=device,
    num_samples=10
)

In [ ]:
def get_predictions_and_references(loader, model, tokenizer, device, max_len):
    model.eval()
    predictions, references = [], []
    complexities = []   # ← YENİ: her cümlenin morfolojik karmaşıklığı

    sos_idx = tokenizer.token_to_id('<SOS>')
    eos_idx = tokenizer.token_to_id('<EOS>')
    pad_idx = tokenizer.token_to_id('<PAD>')

    from torch_geometric.data import Batch
    progress_bar = tqdm(loader, desc='Generating Predictions')

    with torch.no_grad():
        for data in progress_bar:
            if data is None: continue
            src_bpe, src_bpe_word_ids, src_graph, trg_bpe = data
            graph_list = src_graph.to_data_list() if hasattr(src_graph, 'to_data_list') else src_graph

            for i in range(src_bpe.shape[0]):
                single_src_bpe   = src_bpe[i:i+1].to(device)
                single_word_ids  = src_bpe_word_ids[i:i+1].to(device)
                single_graph_obj = graph_list[i]                     # ← karmaşıklık için sakla
                single_graph     = Batch.from_data_list([single_graph_obj]).to(device)

                pred_ids = model.generate(
                    src_graph=single_graph, src_bpe=single_src_bpe,
                    src_bpe_word_ids=single_word_ids,
                    sos_idx=sos_idx, eos_idx=eos_idx, pad_idx=pad_idx,
                    max_len=max_len, beam_size=3, repetition_penalty=1.0
                )

                predicted_text = tokenizer.decode(pred_ids, skip_special_tokens=True)
                predicted_text = predicted_text.replace(" .", ".").replace(" ,", ",").replace(" ?", "?").replace(" !", "!")
                reference_text = tokenizer.decode(trg_bpe[i].cpu().tolist(), skip_special_tokens=True)
                reference_text = reference_text.replace(" .", ".").replace(" ,", ",").replace(" ?", "?").replace(" !", "!")

                predictions.append(predicted_text)
                references.append([reference_text])

                # ← YENİ: morfolojik karmaşıklık = morfem düğüm sayısı / kelime sayısı
                num_nodes = single_graph_obj.x.size(0) - 1              # PAD düğümü çıkar
                num_words = int(single_graph_obj.word_batch.max().item())
                complexities.append(num_nodes / num_words if num_words > 0 else 0.0)

    return predictions, references, complexities   # ← 3'lü döndür

In [ ]:
#HÜCRE 32
#bertscore için yeni
# Bu hücre, test setinizin büyüklüğüne göre zaman alacaktır.

import evaluate
import numpy as np

print("Test seti için çeviriler oluşturuluyor...")

predictions, references, complexities = get_predictions_and_references(
    loader=test_loader, model=model, tokenizer=tokenizer,
    device=device, max_len=MAX_LEN
)

# İlk birkaç örneği görelim
print("\n--- Sample Translations ---")
for i in range(10):
    print(f"Prediction {i+1}: {predictions[i]}")
    print(f"Reference  {i+1}: {references[i][0]}")
    print("-" * 25)
print(f"\n{len(predictions)} adet çeviri başarıyla oluşturuldu.")

# *** DÜZELTME: References formatını sacrebleu'nun beklediği şekle dönüştür ***
references_flat = [r[0] for r in references]   # [["ref1"], ["ref2"]] → ["ref1", "ref2"]
references_for_sacrebleu = [references_flat]   # [["ref1", "ref2", ...]]

# Önce Sacrebleu metrikleri
print("\n--- Evaluation Metrics ---")
bleu = sacrebleu.corpus_bleu(predictions, references_for_sacrebleu)
print(f"BLEU Score: {bleu.score:.2f}")

chrf = sacrebleu.corpus_chrf(predictions, references_for_sacrebleu)
print(f"chrF Score: {chrf.score:.2f}")

ter = sacrebleu.corpus_ter(predictions, references_for_sacrebleu)
print(f"TER Score: {ter.score:.2f}")

# BERTScore zaten doğru kullanıyor (flat list bekliyor):
bertscore_metric = evaluate.load("bertscore")
bert_results = bertscore_metric.compute(
    predictions=predictions,
    references=references_flat,   # ← flat liste, [r[0] for r in references] zaten yapıldı
    lang="en"
)
bert_f1_avg = np.mean(bert_results['f1'])
print(f"BERTScore (Avg F1): {bert_f1_avg:.4f}")

In [ ]:
# MORFOLOJİK KARMAŞIKLIK ANALİZİ
import numpy as np
import sacrebleu
import pandas as pd
import matplotlib.pyplot as plt

references_flat = [r[0] for r in references]

print(f"Standart Transformer")
# Dağılımı önce gör — binleri gerçek dağılıma göre ayarla
print(f"Karmaşıklık — ort: {np.mean(complexities):.2f}, medyan: {np.median(complexities):.2f}, "
      f"min: {np.min(complexities):.2f}, max: {np.max(complexities):.2f}")

bins = [(1.0, 2.5), (2.5, 3.0), (3.0, 3.5), (3.5, 100.0)]
bin_labels = ['< 2.5', '[2.5, 3.0)', '[3.0, 3.5)', '≥ 3.5']

results = []
for (low, high), label in zip(bins, bin_labels):
    idx = [i for i, c in enumerate(complexities) if low <= c < high]
    if not idx:
        continue
    b_preds = [predictions[i] for i in idx]
    b_refs  = [references_flat[i] for i in idx]
    score = sacrebleu.corpus_bleu(b_preds, [b_refs]).score
    note = f"(only {len(idx)} sentences)" if len(idx) < 20 else ""
    results.append({'Complexity': label, 'Count': len(idx),
                    'BLEU': round(score, 2), 'Note': note})

df_complexity = pd.DataFrame(results)
print("\n--- BLEU BY MORPHOLOGICAL COMPLEXITY ---")
print(df_complexity.to_string(index=False))

plt.figure(figsize=(8, 5))
plt.bar(df_complexity['Complexity'], df_complexity['BLEU'],
        color='skyblue', edgecolor='black')
plt.ylabel('BLEU Score')
plt.xlabel('Average Morphemes per Word')
plt.savefig(f"{CHECKPOINT_DIR}/bleu_vs_morphology.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

labels = ['< 2.5', '[2.5, 3.0)', '[3.0, 3.5)', '≥ 3.5']
baseline = [58.03, 61.43, 61.25, 63.49]
morphgat = [57.85, 62.05, 61.62, 62.97]

x = np.arange(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, baseline, width, label='Baseline (6 layers)', color='coral', edgecolor='white')
ax.bar(x + width/2, morphgat, width, label='MorphGAT', color='skyblue', edgecolor='white')
ax.set_xlabel('Average Morphemes per Word')
ax.set_ylabel('BLEU Score')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(50, 66)
ax.legend()
plt.savefig(f"{CHECKPOINT_DIR}/bleu_morphology_comparison.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#  HÜCRE 31: CÜMLE UZUNLUĞUNA GÖRE BLEU ANALİZİ
def analyze_bleu_by_length(predictions, references):
    references_flat = [r[0] for r in references]
    lengths = [len(ref.split()) for ref in references_flat]

    buckets = [(0, 10), (10, 20), (20, 30), (30, 50), (50, 100)]
    results = []

    for low, high in buckets:
        b_preds = [p for p, l in zip(predictions, lengths) if low < l <= high]
        b_refs  = [r for r, l in zip(references_flat, lengths) if low < l <= high]

        if b_preds:
            score = sacrebleu.corpus_bleu(b_preds, [b_refs]).score   # ← [b_refs]
            results.append({'Range': f'{low}-{high}', 'Count': len(b_preds), 'BLEU': round(score, 2)})


    df_len = pd.DataFrame(results)
    print("\n--- TABLE: BLEU SCORE BY SENTENCE LENGTH ---")
    print(df_len)

    plt.figure(figsize=(8, 5))
    plt.bar(df_len['Range'], df_len['BLEU'], color='skyblue', edgecolor='black')
    plt.title('BLEU Score vs. Sentence Length')
    plt.ylabel('BLEU Score')
    plt.xlabel('Number of Words in Source Sentence')
    plt.savefig(f"{CHECKPOINT_DIR}/bleu_vs_length.png")
    plt.show()

analyze_bleu_by_length(predictions, references)

In [ ]:
#HÜCRE 35
import nltk

# METEOR için gerekli olan 'wordnet', 'punkt' ve 'omw-1.4' paketlerini indir
# Open Multilingual Wordnet (omw-1.4) İngilizce dışındaki diller için de gereklidir.
# Eklemenin bir zararı olmaz.

REQUIRED_PACKAGES = ['wordnet', 'punkt', 'omw-1.4']

print("Checking for NLTK packages...")
for package in REQUIRED_PACKAGES:
    try:
        # Paketin varlığını kontrol et
        nltk.data.find(f'corpora/{package}' if package in ['wordnet', 'omw-1.4'] else f'tokenizers/{package}')
        print(f"  - '{package}' package is already downloaded.")
    except LookupError:
        # Eğer paket bulunamazsa, LookupError hatası verir. Bu hatayı yakala ve indir.
        print(f"  - '{package}' package not found. Downloading now...")
        nltk.download(package)
        print(f"  - '{package}' downloaded successfully.")

print("\nAll required NLTK packages are ready.")

In [ ]:
# HÜCRE 36 (DÜZELTİLMİŞ: IEEE ACCESS STANDART TABLOSU + SİSTEM TÜKETİMİ)
import evaluate
import numpy as np
import pandas as pd
import torch
import time
from nltk.translate.meteor_score import meteor_score

# =====================================================================
# 1. REFERANS FORMATINI DÜZELT
# =====================================================================
# references = [["ref1"], ["ref2"], ...] formatından flat ve transposed forma
references_flat       = [r[0] for r in references]            # ["ref1", "ref2", ...]
references_sacrebleu  = [references_flat]                     # [["ref1", "ref2", ...]]

# =====================================================================
# 2. METRİKLER (sacrebleu için doğru format)
# =====================================================================
print("Computing sacrebleu metrics...")
bleu = sacrebleu.corpus_bleu(predictions, references_sacrebleu)
chrf = sacrebleu.corpus_chrf(predictions, references_sacrebleu)
ter  = sacrebleu.corpus_ter(predictions,  references_sacrebleu)

# =====================================================================
# 3. BERTScore
# =====================================================================
print("Computing BERTScore...")
bertscore  = evaluate.load("bertscore")
bs_results = bertscore.compute(
    predictions=predictions,
    references=references_flat,
    lang="en"
)
avg_bert_f1 = np.mean(bs_results['f1'])
avg_bert_p  = np.mean(bs_results['precision'])
avg_bert_r  = np.mean(bs_results['recall'])

# =====================================================================
# 4. METEOR
# =====================================================================
print("Computing METEOR...")
m_scores   = [meteor_score([r.split()], p.split())
              for p, r in zip(predictions, references_flat)]
avg_meteor = np.mean(m_scores)

# =====================================================================
# 5. SİSTEM KAYNAK TÜKETİMİ (Eğitim İstatistikleri)
# =====================================================================
history_df       = pd.read_csv(f"{CHECKPOINT_DIR}/metrics_history.csv")
avg_time_epoch   = history_df['duration'].mean()
total_train_time = history_df['duration'].sum()
total_epochs     = len(history_df)

# Model parametre sayıları
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Inference hızı ölçümü (test setinin küçük bir kısmında)
print("Measuring inference speed...")
model.eval()
inference_times = []
sample_count    = 0
MAX_INFERENCE_SAMPLES = 50

with torch.no_grad():
    for data in test_loader:
        if sample_count >= MAX_INFERENCE_SAMPLES: break
        src_bpe, src_bpe_word_ids, src_graph, trg_bpe = data
        from torch_geometric.data import Batch
        graph_list = src_graph.to_data_list() if hasattr(src_graph, 'to_data_list') else src_graph

        for i in range(src_bpe.size(0)):
            if sample_count >= MAX_INFERENCE_SAMPLES: break

            single_graph    = Batch.from_data_list([graph_list[i]]).to(device)
            single_src_bpe  = src_bpe[i:i+1].to(device)
            single_word_ids = src_bpe_word_ids[i:i+1].to(device)

            t0 = time.time()
            _ = model.generate(
                src_graph=single_graph,
                src_bpe=single_src_bpe,
                src_bpe_word_ids=single_word_ids,
                sos_idx=2, eos_idx=3, pad_idx=0,
                max_len=100, beam_size=3, repetition_penalty=1.0
            )
            inference_times.append(time.time() - t0)
            sample_count += 1

avg_inference_time = np.mean(inference_times) * 1000   # ms cinsinden

# GPU bellek
if torch.cuda.is_available():
    gpu_name        = torch.cuda.get_device_name(0)
    max_vram_gb     = history_df['duration'].mean()  # placeholder
    # Gerçek max VRAM — eğitim sırasında ölçülmüştü, son epoch'tan al
    current_vram_gb = torch.cuda.max_memory_allocated() / (1024**3)
else:
    gpu_name        = "CPU"
    current_vram_gb = 0.0

# =====================================================================
# 6. FİNAL TABLO
# =====================================================================
print("\n" + "="*70)
print(f"         FINAL RESULTS  —  {DATASET}  /  {MODE_SUFFIX}")
print("="*70)

# Çeviri kalitesi metrikleri
quality_metrics = pd.DataFrame({
    "Metric": [
        "BLEU (↑)",
        "chrF (↑)",
        "METEOR (↑)",
        "TER (↓)",
        "BERTScore F1 (↑)",
        "BERTScore Precision (↑)",
        "BERTScore Recall (↑)",
    ],
    "Score": [
        f"{bleu.score:.2f}",
        f"{chrf.score:.2f}",
        f"{avg_meteor:.4f}",
        f"{ter.score:.2f}",
        f"{avg_bert_f1:.4f}",
        f"{avg_bert_p:.4f}",
        f"{avg_bert_r:.4f}",
    ]
})
print("\n📊 TRANSLATION QUALITY METRICS")
print(quality_metrics.to_string(index=False))

# Sistem kaynak tüketimi
system_metrics = pd.DataFrame({
    "Resource": [
        "Total Parameters",
        "Trainable Parameters",
        "Total Epochs",
        "Avg. Time / Epoch",
        "Total Training Time",
        "Avg. Inference Time / Sentence",
        "Peak VRAM",
        "GPU",
    ],
    "Value": [
        f"{total_params:,}",
        f"{trainable_params:,}",
        f"{total_epochs}",
        f"{avg_time_epoch:.1f} s ({avg_time_epoch/60:.1f} min)",
        f"{total_train_time:.0f} s ({total_train_time/3600:.2f} hours)",
        f"{avg_inference_time:.1f} ms",
        f"{current_vram_gb:.2f} GB",
        gpu_name,
    ]
})
print("\n💻 SYSTEM RESOURCE CONSUMPTION")
print(system_metrics.to_string(index=False))

print("="*70)

# =====================================================================
# 7. SONUÇLARI CSV OLARAK KAYDET
# =====================================================================
results_summary = {
    "dataset":            DATASET,
    "mode":               MODE_SUFFIX,
    "bleu":               bleu.score,
    "chrf":               chrf.score,
    "meteor":             avg_meteor,
    "ter":                ter.score,
    "bertscore_f1":       avg_bert_f1,
    "bertscore_precision": avg_bert_p,
    "bertscore_recall":   avg_bert_r,
    "total_params":       total_params,
    "trainable_params":   trainable_params,
    "total_epochs":       total_epochs,
    "avg_time_per_epoch_s": avg_time_epoch,
    "total_train_time_h": total_train_time / 3600,
    "avg_inference_ms":   avg_inference_time,
    "peak_vram_gb":       current_vram_gb,
    "gpu":                gpu_name,
}

results_df  = pd.DataFrame([results_summary])
results_csv = f"{CHECKPOINT_DIR}/final_results.csv"
results_df.to_csv(results_csv, index=False)
print(f"\n✅ Sonuçlar kaydedildi: {results_csv}")

In [ ]:
# HÜCRE 32-EK: bu modelin tahminlerini diske yaz (significance testi için)
import json
print("HÜCRE A Çıktısı")
tag = f"{MODE_SUFFIX}"   # "Dual" (MorphGAT) ya da "Standart" (baseline)
with open(f"{CHECKPOINT_DIR}/preds_{tag}.json", "w") as f:
    json.dump({"predictions": predictions,
               "references_flat": references_flat}, f)
print(f"✅ Kaydedildi: preds_{tag}.json  ({len(predictions)} cümle)")

# SacreBLEU imzası (hakem H3-¶6) — metrik sınıfı üzerinden
from sacrebleu.metrics import BLEU
_bleu = BLEU()
_score = _bleu.corpus_score(predictions, [references_flat])
print("BLEU:", _score.score)
print("SacreBLEU signature:", _bleu.get_signature())


from sacrebleu.metrics import CHRF, TER
_chrf, _ter = CHRF(), TER()
_chrf.corpus_score(predictions, [references_flat])   # önce çalıştır
_ter.corpus_score(predictions, [references_flat])    # önce çalıştır
print("chrF signature:", _chrf.get_signature())
print("TER signature :", _ter.get_signature())

In [ ]:
# HÜCRE 32-EK: morfolojik karmaşıklık binleri — sayı + %95 GA + uzunluk kontrolü
import numpy as np, sacrebleu, pandas as pd
print("HÜCRE C:")
print("Ana model:")


# 1) predictions ile aynı sırada karmaşıklık + uzunluk üret
complexities, src_lengths = [], []
for data in test_loader:
    src_bpe, src_bpe_word_ids, src_graph, trg_bpe = data
    glist = src_graph.to_data_list()
    for i in range(src_bpe.size(0)):
        g = glist[i]
        n_words = int(g.word_batch.max().item())          # PAD düğümü hariç kelime sayısı
        n_nodes = g.x.size(0) - 1                          # PAD düğümünü çıkar
        complexities.append(n_nodes / max(n_words, 1))     # kelime başına morfolojik birim
        src_lengths.append(n_words)

assert len(complexities) == len(predictions)

def bleu_ci(idx_list, B=500, seed=0):
    idx = np.array(idx_list)
    pts = [predictions[i] for i in idx]; rfs = [references_flat[i] for i in idx]
    score = sacrebleu.corpus_bleu(pts, [rfs]).score
    rng = np.random.default_rng(seed); n = len(idx); boot = []
    for _ in range(B):
        s = rng.integers(0, n, n)
        boot.append(sacrebleu.corpus_bleu([pts[j] for j in s], [[rfs[j] for j in s]]).score)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return score, lo, hi, n

comp = np.array(complexities)
bins = [(1.0,1.5),(1.5,2.0),(2.0,2.5),(2.5,99)]
rows = []
for lo_c, hi_c in bins:
    idx = [i for i,c in enumerate(comp) if lo_c <= c < hi_c]
    if idx:
        s, lo, hi, cnt = bleu_ci(idx)
        rows.append({"Karmasiklik": f"{lo_c}-{hi_c}", "N": cnt,
                     "BLEU": round(s,2), "95% GA": f"[{lo:.1f}, {hi:.1f}]"})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# HÜCRE 32-EK: en karmaşık cümlelerden örnek çeviriler (H2-7)
print("HÜCRE D:")
print("Ana model:")


order = np.argsort(complexities)[::-1][:8]   # en yüksek karmaşıklık
for i in order:
    print(f"[karmaşıklık {complexities[i]:.2f}]")
    print("  TAHMİN  :", predictions[i])
    print("  REFERANS:", references_flat[i])
    print("-"*70)

In [ ]:
import json, numpy as np, sacrebleu, matplotlib.pyplot as plt

D = "/content/drive/MyDrive/Tez_Checkpoints/EN2TR_CrossAttention"
base  = json.load(open(f"/content/drive/MyDrive/Tez_Checkpoints/EN2TR_Standart_N6/EN2TR_Standart/preds_Standart.json"))
morph = json.load(open(f"{D}/EN2TR_Dual/preds_Dual.json"))
refs = base["references_flat"]
assert refs == morph["references_flat"]

# Karmaşıklık (test_loader sırası = prediction sırası)
complexities = []
for data in test_loader:
    src_bpe, src_bpe_word_ids, src_graph, trg_bpe = data
    for g in src_graph.to_data_list():
        n_words = int(g.word_batch.max().item())
        n_nodes = g.x.size(0) - 1
        complexities.append(n_nodes / max(n_words, 1))
complexities = np.array(complexities)
assert len(complexities) == len(refs)

bins = [("<2.5", -np.inf, 2.5), ("[2.5,3.0)", 2.5, 3.0),
        ("[3.0,3.5)", 3.0, 3.5), ("≥3.5", 3.5, np.inf)]

def bleu_ci(preds, idx, B=500, seed=0):
    pts=[preds[i] for i in idx]; rfs=[refs[i] for i in idx]
    s=sacrebleu.corpus_bleu(pts,[rfs]).score
    rng=np.random.default_rng(seed); n=len(idx); bt=[]
    for _ in range(B):
        r=rng.integers(0,n,n)
        bt.append(sacrebleu.corpus_bleu([pts[j] for j in r],[[rfs[j] for j in r]]).score)
    lo,hi=np.percentile(bt,[2.5,97.5]); return s,lo,hi

labels=[]; N=[]; bs=[]; be_lo=[]; be_hi=[]; ms=[]; me_lo=[]; me_hi=[]
print(f"{'Bin':<12}{'N':>6}{'Baseline':>12}{'MorphGAT':>12}")
for name, lo, hi in bins:
    idx=[i for i,c in enumerate(complexities) if lo <= c < hi]
    sb,lb,hb = bleu_ci(base["predictions"], idx)
    sm,lm,hm = bleu_ci(morph["predictions"], idx)
    labels.append(name); N.append(len(idx))
    bs.append(sb); be_lo.append(sb-lb); be_hi.append(hb-sb)
    ms.append(sm); me_lo.append(sm-lm); me_hi.append(hm-sm)
    print(f"{name:<12}{len(idx):>6}{sb:>8.2f} [{lb:.1f},{hb:.1f}]  {sm:>8.2f} [{lm:.1f},{hm:.1f}]")

x=np.arange(len(labels)); w=0.38
fig,ax=plt.subplots(figsize=(9,6))
ax.bar(x-w/2, bs, w, yerr=[be_lo,be_hi], capsize=4, label="Baseline (6L)", color="#2c3e50")
ax.bar(x+w/2, ms, w, yerr=[me_lo,me_hi], capsize=4, label="MorphGAT",     color="#5dade2")
for i,n in enumerate(N):
    ax.text(x[i], 3, f"N={n}", ha="center", fontsize=9, color="white", fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_xlabel("Morphological complexity (units per word)")
ax.set_ylabel("BLEU"); ax.set_ylim(0, 75)
ax.legend(); ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig("Baseline_vs_CrossAttn.png", dpi=300); plt.show()

In [ ]:
# BAĞIMSIZ HÜCRE: Temiz test (eğitimle örtüşen kaynakları çıkar) — her model için
import pandas as pd, sacrebleu
norm = lambda s: ' '.join(str(s).lower().split())
print("C. Sızıntı için ek analiz")

D = '/content/drive/MyDrive/Tez_Veri/EN2TR'
train_src = set(pd.read_csv(f'{D}/train.tsv', sep='\t', names=['tr','en'],
                            on_bad_lines='skip').dropna()['tr'].map(norm))
test_df = pd.read_csv(f'{D}/test.tsv', sep='\t', names=['tr','en'],
                      on_bad_lines='skip').dropna().reset_index(drop=True)

assert len(test_df) == len(predictions), \
    f"Sıralama uyuşmuyor: test.tsv={len(test_df)} vs predictions={len(predictions)}"

keep = [norm(s) not in train_src for s in test_df['tr']]
idx  = [i for i, k in enumerate(keep) if k]
clean_preds = [predictions[i] for i in idx]
clean_refs  = [references_flat[i] for i in idx]

full  = sacrebleu.corpus_bleu(predictions,  [references_flat]).score
clean = sacrebleu.corpus_bleu(clean_preds,  [clean_refs]).score
print(f"Tam test  : {len(predictions)} cümle | BLEU {full:.2f}")
print(f"Temiz test: {len(idx)} cümle | BLEU {clean:.2f}")
print(f"Fark      : {clean - full:+.2f} BLEU")

In [ ]:
print("kontrol f çıktısı")
print(model.graph_encoder.gat1.add_self_loops)   # True bekliyoruz

In [ ]:
# 4 eğitim sonrası — comparison table:
import pandas as pd

results = []
for ds in ["EN2TR", "WMT18"]:
    for mode in ["Standart", "Dual"]:
        path = f"/content/drive/MyDrive/Tez_Checkpoints/{ds}_{mode}/final_results.csv"
        if os.path.exists(path):
            results.append(pd.read_csv(path))

comparison_df = pd.concat(results, ignore_index=True)
print(comparison_df)

ValueError: No objects to concatenate

In [ ]:
import pandas as pd
import os

results = []
base_checkpoints_root = "/content/drive/MyDrive/Tez_Checkpoints/"

# Ensure the base directory exists
if not os.path.exists(base_checkpoints_root):
    print(f"Warning: Base checkpoint directory not found: {base_checkpoints_root}")
    print("No results to compare.")
else:
    for ds in ["EN2TR", "WMT18"]:
        for mode in ["Standart", "Dual"]:
            target_config_folder_name = f"{ds}_{mode}" # e.g., EN2TR_Dual

            found_path = None
            # Iterate through all immediate subdirectories of base_checkpoints_root
            # (e.g., 'EN2TR_Dual_N6', 'WMT18_Standart_N5', etc.)
            for run_id_folder in os.listdir(base_checkpoints_root):
                full_run_id_path = os.path.join(base_checkpoints_root, run_id_folder)
                if os.path.isdir(full_run_id_path):
                    # Check if this run_id_folder contains the target_config_folder_name
                    potential_config_path = os.path.join(full_run_id_path, target_config_folder_name)
                    final_result_path = os.path.join(potential_config_path, "final_results.csv")

                    if os.path.exists(final_result_path):
                        found_path = final_result_path
                        break # Found the correct path for this (ds, mode)

            if found_path:
                df = pd.read_csv(found_path)
                # Add 'dataset' and 'mode' columns for easy identification in the comparison table
                df['dataset'] = ds
                df['mode'] = mode
                results.append(df)
            else:
                print(f"Warning: Results file not found for {ds} {mode}. Expected in structure: {base_checkpoints_root}RUN_ID/{target_config_folder_name}/final_results.csv")

    if results:
        comparison_df = pd.concat(results, ignore_index=True)
        print(comparison_df)
    else:
        print("No results found to create a comparison table.")